# Fixed points, bistability, and the input as a parameter

In `7-rcp-RNN-as-Sequential2D.ipynb` we built an RNN as an iterated block map

$$z_{t+1} = (A \circ b \circ M)^{K} \circ \mathrm{Inject}_{t+1} \circ z_t$$

and saw the internal map settle down as $K$ grew. This notebook asks what it is
settling *to*, and the answer turns out to be the entire point of the
construction.

The trick that makes this visible is to shrink the hidden state to **one
dimension**. Everything then happens on a line, every claim can be drawn, and
the machinery is the ordinary theory of one-dimensional maps — cobwebs, fixed
points, stability, folds. The model is not a toy analogue of the real one; it is
the same code with `hidden_size = 1`.

What we will establish:

1. With a weak recurrence the map has **one** stable fixed point. The state
   forgets its history: no memory.
2. With a strong recurrence it has **two** stable fixed points. Which one you
   land in is set by where you started: one bit of memory, stored in the
   *identity of an attractor* rather than in a decaying transient.
3. Changing the **input** moves and eventually destroys those fixed points. The
   input is not a nudge to the state — it reshapes the landscape the state is
   moving in.

Point 3 is why an RNN written this way is more than a change of notation.

In [1]:
import numpy as np
import torch
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots

# Load plotly.js from a CDN instead of embedding a copy in every figure, which
# otherwise makes this notebook several megabytes.  Needs a network connection to
# view the plots; switch to 'notebook' to embed them.
pio.renderers.default = 'notebook_connected'

from iterativennsimple.Sequential2DRNN import Sequential2DRNN

torch.manual_seed(0)

## 1. One hidden unit

With `hidden_size = 1` and `input_size = 1` the internal map $A \circ b \circ M$
acting on the hidden slot is just

$$h \;\longmapsto\; \tanh\!\big(w_{hh}\,h + w_{xh}\,x + b\big)$$

Since $x$ is held fixed across the internal iterations (that is what the $I$ in
the corner of $M$ is for), the input enters only through the constant

$$c \;=\; w_{xh}\,x + b$$

so the map we must understand is the one-parameter family
$\;f_c(h) = \tanh(w_{hh}h + c)$.

Let us check that this really is what the module computes, rather than taking it
on faith.

In [2]:
def one_unit_model(w_hh, w_xh, b, K=1):
    """A Sequential2DRNN with a single hidden unit and prescribed weights."""
    linear = lambda i, o: torch.nn.Linear(i, o, bias=False)
    W_xh, W_hh, W_hy = linear(1, 1), linear(1, 1), linear(1, 1)
    with torch.no_grad():
        W_xh.weight.fill_(w_xh)
        W_hh.weight.fill_(w_hh)
        W_hy.weight.fill_(1.0)
    model = Sequential2DRNN.from_3x3(
        input_size=1, output_size=1, hidden_size=1,
        W_xh=W_xh, W_hh=W_hh, W_hy=W_hy,
        A_h=torch.nn.Tanh(), A_y=torch.nn.Identity(),
        K=K, batch_first=True)
    with torch.no_grad():
        model.b['2'].fill_(b)          # slot 2 is h; slot 1 is y
        model.b['1'].zero_()
    return model


w_hh, w_xh, b = 1.8, 1.0, 0.0
model = one_unit_model(w_hh, w_xh, b)

x_value, h_value = 0.3, -0.4
z = model.internal_step([torch.tensor([[x_value]]),
                         None,
                         torch.tensor([[h_value]])])

by_formula = np.tanh(w_hh * h_value + w_xh * x_value + b)
print(f'from the module : {z[2].item():.10f}')
print(f'from the formula: {by_formula:.10f}')
assert abs(z[2].item() - by_formula) < 1e-6

from the module : -0.3969303966
from the formula: -0.3969304320


Good. From here we can work with the scalar map directly and know we are
describing the model.

In [3]:
def f(h, w_hh, c):
    """The internal map restricted to the hidden slot."""
    return np.tanh(w_hh * h + c)


def fixed_points(w_hh, c, n_grid=4001, tol=1e-9):
    """All solutions of f(h) = h.

    Iterating from random starts would only ever find the *stable* fixed points.
    The unstable one separates the basins, so we want it too, which means solving
    g(h) = f(h) - h = 0 directly.

    Two cases, and conflating them is an easy way to count wrong: g may cross zero
    *between* grid points (bisect), or land exactly *on* one (h = 0 whenever
    c = 0, since tanh is odd).  Testing sign changes with `sign(a) != sign(b)`
    reports an exact zero as two crossings, so use a strict product test for the
    former and collect the latter separately.
    """
    grid = np.linspace(-1.0, 1.0, n_grid)
    g = f(grid, w_hh, c) - grid

    roots = list(grid[np.abs(g) < 1e-13])                  # landed exactly on a root
    for i in np.where(g[:-1] * g[1:] < 0)[0]:              # strict sign change
        low, high = grid[i], grid[i + 1]
        for _ in range(80):                                # plain bisection
            mid = 0.5 * (low + high)
            if (f(low, w_hh, c) - low) * (f(mid, w_hh, c) - mid) > 0:
                low = mid
            else:
                high = mid
        roots.append(0.5 * (low + high))

    roots = np.sort(np.array(roots))
    return np.array([r for i, r in enumerate(roots)
                     if i == 0 or r - roots[i - 1] > tol])


def multiplier(h_star, w_hh):
    """f'(h*) -- the map contracts near h* when |f'| < 1, so the point is stable.

    d/dh tanh(w h + c) = w * sech^2(w h + c) = w * (1 - tanh^2) = w * (1 - h*^2)
    at a fixed point, since h* = tanh(w h* + c).
    """
    return w_hh * (1.0 - h_star ** 2)


def classify(h_star, w_hh, tol=1e-9):
    """Linear stability from the multiplier -- with the marginal case named.

    |f'| < 1 stable, |f'| > 1 unstable, and |f'| = 1 *non-hyperbolic*, where
    linearisation says nothing at all.  That third case is not a technicality to
    be rounded away: it is exactly where bifurcations happen, so a tutorial that
    quietly reports it as "unstable" would be hiding the interesting point.
    """
    m = abs(multiplier(h_star, w_hh))
    if abs(m - 1.0) < tol:
        return "non-hyperbolic"
    return "stable" if m < 1 else "UNSTABLE"


def fold_location(w_hh):
    """Exactly where the saddle-node happens, for w_hh > 1.

    A fold is where a fixed point is about to be born or destroyed: the curve is
    tangent to the diagonal, so f(h) = h and f'(h) = 1 hold together.

        f'(h) = w (1 - h^2) = 1   =>   h = +- sqrt(1 - 1/w)
        h = tanh(w h + c)         =>   c = arctanh(h) - w h

    No sweeping, no grid resolution to worry about.
    """
    h = np.sqrt(1.0 - 1.0 / w_hh)
    return abs(np.arctanh(h) - w_hh * h), h

## 2. Weak recurrence: one attractor, no memory

The standard picture for a one-dimensional map. Plot $f$ against the diagonal;
fixed points are the crossings. Starting anywhere and iterating traces a
staircase — the **cobweb** — into whichever fixed point attracts it.

In [4]:
def cobweb(w_hh, c, starts, n_steps=30):
    """Traces of the iteration, as (h_k, h_{k+1}) staircase segments."""
    traces = []
    for h in starts:
        xs, ys = [h], [0.0]
        for _ in range(n_steps):
            h_next = f(h, w_hh, c)
            xs += [h, h_next]           # up to the curve, then across to the diagonal
            ys += [h_next, h_next]
            h = h_next
        traces.append((xs, ys))
    return traces


def add_cobweb_panel(figure, row, col, w_hh, c, starts):
    """Draw f, the diagonal, the staircases, and the fixed points into one panel.

    Fixed points are marked filled when stable (|f'| < 1) and hollow when not.
    """
    grid = np.linspace(-1, 1, 400)
    figure.add_trace(go.Scatter(x=grid, y=f(grid, w_hh, c), name='f(h)',
                                line=dict(width=3, color='royalblue'),
                                showlegend=(row == 1 and col == 1)), row=row, col=col)
    figure.add_trace(go.Scatter(x=grid, y=grid, name='diagonal',
                                line=dict(dash='dash', color='grey'),
                                showlegend=(row == 1 and col == 1)), row=row, col=col)
    for xs, ys in cobweb(w_hh, c, starts):
        figure.add_trace(go.Scatter(x=xs, y=ys, line=dict(width=1, color='darkorange'),
                                    showlegend=False), row=row, col=col)
    for h_star in fixed_points(w_hh, c):
        stable = classify(h_star, w_hh) == 'stable'
        figure.add_trace(go.Scatter(
            x=[h_star], y=[h_star], mode='markers', showlegend=False,
            marker=dict(size=11, color='black' if stable else 'white',
                        line=dict(width=2, color='black'))), row=row, col=col)


figure = make_subplots(rows=1, cols=2, subplot_titles=(
    'weak recurrence: w_hh = 0.7', 'strong recurrence: w_hh = 1.8'))
add_cobweb_panel(figure, 1, 1, w_hh=0.7, c=0.0, starts=[-0.9, 0.9])
add_cobweb_panel(figure, 1, 2, w_hh=1.8, c=0.0, starts=[-0.9, -0.05, 0.05, 0.9])
figure.update_layout(
    title='Cobwebs for h -> tanh(w_hh h).  Filled = stable, hollow = unstable.',
    template='plotly_white', width=900, height=470)
figure.update_xaxes(title_text='h_k')
figure.update_yaxes(title_text='h_(k+1)', col=1)
figure.show()

Both starting points end at the same place. Formally, $|f'(h)| \le w_{hh} < 1$
everywhere, so the map is a **contraction** and the Banach fixed point theorem
gives a unique fixed point reached from any initial condition.

For the network this is a death sentence for memory: the converged state is the
same regardless of $h_t$, so nothing survives from one token to the next. This is
precisely the collapse measured in `examples/rnn_internal_iterations.py` — and
note it is a statement about iterating a contraction, not about neural networks.

In [5]:
print(f'{"w_hh":>6}  {"fixed points":>40}  multipliers')
for w in [0.5, 0.7, 0.9]:
    pts = fixed_points(w, 0.0)
    mults = [f'{multiplier(p, w):+.2f}' for p in pts]
    print(f'{w:>6}  {str(np.round(pts, 4)):>40}  {mults}')

  w_hh                              fixed points  multipliers
   0.5                                      [0.]  ['+0.50']
   0.7                                      [0.]  ['+0.70']
   0.9                                      [0.]  ['+0.90']


## 3. Strong recurrence: two attractors, one bit of memory

Increase $w_{hh}$ past 1 and the picture changes qualitatively. At $w_{hh} = 1$
the slope of $f$ at the origin equals the slope of the diagonal; beyond that the
curve crosses the diagonal three times.

In [6]:
for w in [0.9, 1.0, 1.1, 1.8]:
    pts = fixed_points(w, 0.0)
    labels = [classify(p, w) for p in pts]
    print(f'w_hh = {w}:  ' + ',  '.join(f'{p:+.4f} ({l})'
                                        for p, l in zip(pts, labels)))

w_hh = 0.9:  +0.0000 (stable)
w_hh = 1.0:  +0.0000 (non-hyperbolic)
w_hh = 1.1:  -0.5029 (stable),  +0.0000 (UNSTABLE),  +0.5029 (stable)
w_hh = 1.8:  -0.9327 (stable),  +0.0000 (UNSTABLE),  +0.9327 (stable)


Three fixed points: two stable, and the one at the origin now **unstable**. The
unstable point is the boundary between the two basins — start left of it and you
fall left, start right and you fall right.

This is the structure worth internalising. The network can now converge — the
iteration settles, so you may stop early and save compute — *and* remember,
because which of the two attractors it settles into is determined by where it
started. One bit, stored in the identity of an attractor. Unlike a decaying
transient, it does not fade with further iteration.

The transition at $w_{hh} = 1$ is a genuine pitchfork bifurcation, made
symmetric by $c = 0$ and by $\tanh$ being odd.

Look at what the table reports at $w_{hh} = 1$ exactly: **non-hyperbolic**. The
multiplier is exactly 1, so linearisation is silent -- it cannot say whether the
point attracts or repels. That is not a gap in the code; it is the defining
feature of a bifurcation point, and settling the question there needs the next
order term. Here $\tanh(h) \approx h - h^3/3$, and the cubic is stabilising, so
the origin still attracts -- algebraically rather than exponentially, which is
why convergence at $w_{hh} = 1$ is so much slower than at $w_{hh} = 0.9$.

## 4. The input reshapes the landscape

Now the part that matters for the architecture. The input enters through
$c = w_{xh}x + b$, and $c$ shifts the curve vertically. Watch what a nonzero
input does to the same strong recurrence:

In [7]:
figure = make_subplots(rows=1, cols=3,
                       subplot_titles=[f'c = {c}' for c in (0.0, 0.3, 0.6)])
for panel, c in enumerate([0.0, 0.3, 0.6]):
    add_cobweb_panel(figure, 1, panel + 1, w_hh=1.8, c=c, starts=[-0.9, 0.9])
figure.update_layout(
    title='Raising the input term c lifts the curve until two fixed points collide',
    template='plotly_white', width=1000, height=420)
figure.update_xaxes(title_text='h_k')
figure.update_yaxes(title_text='h_(k+1)', col=1)
figure.show()

At $c = 0$ there are two stable fixed points. As $c$ grows the curve lifts, the
lower stable point and the unstable point slide toward each other, and at some
critical $c$ they **collide and annihilate** — leaving one attractor. This
collision of a stable and an unstable fixed point is a saddle-node (fold)
bifurcation, and here, in one dimension, the term is meant literally.

Sweeping $c$ and plotting every fixed point gives the fold diagram:

In [8]:
c_values = np.linspace(-1.2, 1.2, 400)
stable_c, stable_h, unstable_c, unstable_h = [], [], [], []
for c in c_values:
    for h_star in fixed_points(1.8, c):
        if classify(h_star, 1.8) == 'stable':
            stable_c.append(c); stable_h.append(h_star)
        else:
            unstable_c.append(c); unstable_h.append(h_star)

figure = go.Figure()
figure.add_trace(go.Scatter(x=stable_c, y=stable_h, mode='markers', name='stable',
                            marker=dict(size=3, color='black')))
figure.add_trace(go.Scatter(x=unstable_c, y=unstable_h, mode='markers',
                            name='unstable',
                            marker=dict(size=3, color='crimson')))
figure.update_layout(
    title='Fixed points of h -> tanh(1.8 h + c) as the input term c varies',
    xaxis_title='c = w_xh x + b   (the input)', yaxis_title='fixed point h*',
    template='plotly_white', width=700, height=500)
figure.show()

The classic S-curve. Between the two folds the system is **bistable** and can
store a bit; outside them it is monostable and cannot. So the input does not
merely push the state around — it decides *whether the memory exists at all*.

This is what is meant by saying the input conditions the landscape rather than
perturbing the state:

* $x$ determines **what the fixed-point set is**.
* $h$ determines **which element of it is reached**.

Both matter, and the first is the more basic: a converged state that ignores $x$
computes nothing at all.

In [9]:
# The folds have a closed form, so there is no need to hunt for them on a grid.
c_fold, h_fold = fold_location(1.8)
print(f'saddle-node at c = +-{c_fold:.6f}, where the fixed point sits at '
      f'h = -+{h_fold:.6f}')
print(f'so the system is bistable for |c| < {c_fold:.4f} and monostable outside.')

# Cross-check against simply counting roots either side of the predicted fold.
for c in [c_fold - 1e-3, c_fold + 1e-3]:
    print(f'  c = {c:.6f}: {len(fixed_points(1.8, c))} fixed points')

saddle-node at c = +-0.395281, where the fixed point sits at h = -+0.666667
so the system is bistable for |c| < 0.3953 and monostable outside.
  c = 0.394281: 3 fixed points
  c = 0.396281: 1 fixed points


## 5. Reading the two sensitivities off the picture

Write $\Phi(x, h)$ for the fixed point reached from $h$ with input $x$ held. The
two quantities that decide whether the model is any good are

$$\frac{\partial \Phi}{\partial x} \quad\text{(does it respond?)}
\qquad
\frac{\partial \Phi}{\partial h} \quad\text{(does it remember?)}$$

Both are visible above. $\partial\Phi/\partial x$ is the slope of a branch in the
fold diagram — and it blows up at a fold, where an infinitesimal change in input
throws the state onto a different branch. $\partial\Phi/\partial h$ is zero
*within* a basin and undefined across the boundary, so memory here is discrete,
not graded.

In [10]:
def phi(h_0, w_hh, c, n_iter=200):
    h = h_0
    for _ in range(n_iter):
        h = f(h, w_hh, c)
    return h


print(f'{"c":>6}  {"Phi(c, h0=-0.9)":>16}  {"Phi(c, h0=+0.9)":>16}  remembers h0?')
for c in [0.0, 0.3, 0.6, 0.9]:
    low, high = phi(-0.9, 1.8, c), phi(0.9, 1.8, c)
    print(f'{c:>6.1f}  {low:>16.4f}  {high:>16.4f}  {abs(high - low) > 1e-3}')

     c   Phi(c, h0=-0.9)   Phi(c, h0=+0.9)  remembers h0?
   0.0           -0.9327            0.9327  True
   0.3           -0.8340            0.9668  True
   0.6            0.9826            0.9826  False
   0.9            0.9907            0.9907  False


At $c = 0$ and $c = 0.3$ the two initial conditions reach different fixed points
— the bit survives. By $c = 0.6$ the fold has happened and both reach the same
place: same map, same weights, input large enough that the memory no longer
exists.

## 6. Back to the full model

Nothing above needed `hidden_size = 1`; it only needed the state space to be
small enough to draw. In higher dimensions the same objects are there and the
same questions apply, with the Jacobian $J_{T_x}$ replacing $f'$ and its spectral
radius replacing $|f'|$. Confirm the one-unit model really does store its bit
across actual sequence steps:

In [11]:
bistable = one_unit_model(w_hh=1.8, w_xh=1.0, b=0.0, K=20)
x_sequence = torch.zeros(2, 6, 1)                 # no input at all after the start

_, h_n = bistable(x_sequence, h_0=torch.tensor([[[-0.9], [0.9]]]))
print(f'h_n starting from -0.9: {h_n[0, 0, 0]:+.4f}')
print(f'h_n starting from +0.9: {h_n[0, 1, 0]:+.4f}')
print('Six tokens and twenty internal iterations each -- 120 applications of the '
      'map -- and the bit is still there.')

h_n starting from -0.9: -0.9327
h_n starting from +0.9: +0.9327
Six tokens and twenty internal iterations each -- 120 applications of the map -- and the bit is still there.


120 applications of a map, no decay, because the information is in *which*
attractor rather than in a magnitude. Contrast with the weak-recurrence model,
where the same experiment loses everything:

In [12]:
contracting = one_unit_model(w_hh=0.7, w_xh=1.0, b=0.0, K=20)
_, h_n = contracting(x_sequence, h_0=torch.tensor([[[-0.9], [0.9]]]))
print(f'h_n from -0.9: {h_n[0, 0, 0]:+.6f}    h_n from +0.9: {h_n[0, 1, 0]:+.6f}')

h_n from -0.9: -0.000000    h_n from +0.9: +0.000000


## Where to go next

- **Two hidden units.** The same programme in the plane: phase portraits,
  spirals, and the possibility of complex multipliers — a discrete Hopf
  (Neimark–Sacker) bifurcation gives invariant circles, so the internal iteration
  need not converge to a point at all. Whether that is useful or merely pretty is
  an open question.
- **Does training find this?** Everything here was set by hand. Train the model
  from `examples/rnn_internal_iterations.py` on a memory task and measure the
  spectral radius of $J_{T_x}$ before and after. If training pushes it past 1 on
  its own, the orthogonal initialisation in that script is a convenience rather
  than a fix.
- **The convergence loss trap.** §10.3 of `OVERVIEW_RNN_SEQUENTIAL_2D.md` argues
  that a naive loss encouraging convergence is minimised by the monostable
  solution — the left-hand picture in section 2, with no memory. Section 3 here
  is what such a loss would destroy.
- **Sparse recurrence.** Replace $W_{hh}$ with `MonarchLinear` or `MaskedLinear`
  and ask what the sparsity pattern does to the number and arrangement of the
  attractors. That is the sparsity question of the paper, asked in the language
  of dynamical systems.